# Sesión 07 laboratorio práctico: Reto de clasificación para puntuación de oportunidad top-price

**Rol de canalización:** Laboratorio de aula de la Sesión 07 opcional, fuera de la ruta de ejecución obligatoria de un extremo a otro.

> Advertencia: El cuaderno de referencia de producción sigue siendo `05_classification_audi_a3_germany.ipynb`.

Esta práctica de laboratorio se ejecuta directamente desde archivos CSV procesados. No requiere credenciales en la nube, acceso BigQuery ni escrituras en el almacén.

**Consume:** archivos CSV procesados ​​con campos de precio, característica y `price_label`.
**Produce:** resultados del aula y archivos CSV de cola de revisión de clasificación opcionales en `reports/`.
**Feeds:** política de umbral en el aula y discusión de transferencia BI, no la tabla de predicción obligatoria BigQuery.

Head Of Data 101 utiliza una idea central: **el canal es el producto**.

En esta práctica de laboratorio, la clasificación predice si un listado pertenece a la categoría del mercado externo `top-price`. La salida del modelo es `probability_top_price`. El artefacto empresarial es una cola de revisión, no una decisión de adquisición automática.

Límite narrativo del curso:

- `price_label` es una etiqueta de mercado observada.
- `top_price` es el objetivo binario derivado de `price_label`.
- `top_price = 1` cuando `price_label == "top-price"`; en caso contrario `top_price = 0`.
- Ni `price_label` ni `price_label_id` ni `top_price` se pueden utilizar como funciones de entrada del modelo.
- Usar el objetivo, o una fuente directa del objetivo, como característica de entrada es una fuga de objetivo.
- La regresión permanece separada y predice `expected_price_eur`.
- BI luego combina el precio real, la brecha expected-price y la probabilidad top-price.

`price_eur` se utiliza como característica de mercado observable en el clasificador principal. El laboratorio también mantiene una ruta de comparación sin precios para que los estudiantes puedan discutir cuánto cambia la señal del precio en el comportamiento del modelo. `actual_price_eur`, cuando está presente como un nombre duplicado de BI, no se utiliza como una característica separada.

**Nota conceptual:** La selección de umbral no es una calibración de probabilidad. En esta práctica de laboratorio elegimos una regla operativa para la capacidad de revisión. No probamos formalmente que una puntuación de 0,80 signifique una frecuencia empírica del 80%.


## Agenda de sesión en vivo de 3 horas

- 00:00-00:20 - Definición de objetivos, filtración y contrato de datos
- 00:20-00:40 - Distribución de etiquetas y línea de base ingenua
- 00:40-01:20 - Regresión logística como clasificador interpretable
- 01:20-01:55 - Comparación de un modelo flexible
- 01:55-02:30 - Política de umbral y equilibrio precisión/recuperación
- 02:30-02:50 - Cola de revision lista para BI
- 02:50-03:00 - Discusión y entrega


## Ruta en vivo mínima viable

Si el tiempo de clase se vuelve escaso, complete el Desafío 0, el Desafío 1, el Desafío 2, el Desafío 4 y finalice con la cola BI dirigida por un instructor del Desafío 5.


## Cómo trabajar

Este no es un curso puro de codificación. Utilice la codificación asistida por IA cuando sea útil, pero asegúrese de poder explicar cada resultado.

Para cada desafío, lea el motivo comercial, complete la pequeña tarea de codificación o inspeccione el resultado dirigido por el instructor y escriba una breve interpretación comercial en inglés.

Busque las etiquetas **Tarea del estudiante**, **Demostración dirigida por un instructor**, **Pausa de discusión** y **Extensión opcional**. Están ahí para proteger el ritmo de clase.


## Desafío 0: contrato de datos de clasificación y límite de fuga

**Tiempo sugerido:** 20 minutos

**Demostración dirigida por un instructor**

**Razón comercial:**  
Antes de modelar, un científico de datos debe saber exactamente qué contrato de conjunto de datos se está consumiendo y de dónde proviene el objetivo.

**Objetivo:**  
Cargue el CSV procesado, valide las columnas requeridas, cree `listing_id` si es necesario, cree `top_price` a partir de `price_label` si es necesario y prepare un marco de datos de modelado.

**Límite de característica:**
- `price_label` es la etiqueta del mercado observado que se utiliza para definir el objetivo.
- `top_price` es el objetivo.
- `price_label`, `price_label_id` y `top_price` no pueden ser funciones.
- `price_eur` es observable en el mercado y está permitido como característica.
- Aún vale la pena discutir `price_eur` porque puede dominar el modelo.

**Pausa de discusión:**
- ¿`top_price` ya está presente o se creó a partir de `price_label`?
- ¿Por qué `price_eur` es diferente de la fuga objetivo?
- ¿Por qué la definición del objetivo debería fallar estrepitosamente en lugar de ser adivinada?


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.style.use("default")


def find_repo_root(start):
    for path in [start] + list(start.parents):
        if (path / ".git").exists() or (path / "config" / "project_config.yaml").exists():
            return path
    return start


def load_project_config(config_path):
    config = {}
    if not config_path.exists():
        return config
    for raw_line in config_path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(("'", '"')) and value.endswith(("'", '"')):
            value = value[1:-1]
        elif value.lower() in ("true", "false"):
            value = value.lower() == "true"
        else:
            try:
                value = int(value)
            except ValueError:
                try:
                    value = float(value)
                except ValueError:
                    pass
        config[key] = value
    return config


def relative_path(path):
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def csv_missing_required_columns(path, required_columns):
    try:
        columns = pd.read_csv(path, nrows=0).columns
    except Exception as exc:
        return required_columns, str(exc)
    missing = [column for column in required_columns if column not in columns]
    return missing, None


def csv_contains_required_columns(path, required_columns):
    missing, error = csv_missing_required_columns(path, required_columns)
    return not missing and error is None


def newest_csv_in_folder(folder):
    if not folder.exists():
        return None
    csv_files = [path for path in folder.glob("*.csv") if path.is_file()]
    if not csv_files:
        return None
    return max(csv_files, key=lambda path: path.stat().st_mtime)


def newest_data_csv_with_required_columns(data_root, required_columns):
    if not data_root.exists():
        return None
    candidates = []
    for path in data_root.rglob("*.csv"):
        if path.is_file() and csv_contains_required_columns(path, required_columns):
            candidates.append(path)
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)


REQUIRED_COLUMNS = ["price_label", "price_eur", "mileage_km", "age_years", "power_hp"]
SESSION_07_FULL_CSV = "autoscout24_listings_processed_audi_a3_germany_20251228_205210.csv"
OPTIONAL_COLUMNS = [
    "listing_id",
    "make",
    "model",
    "brand",
    "fuel_type",
    "listing_country",
    "price_eur",
    "actual_price_eur",
    "registration_year",
    "registration_month",
    "price_outlier_iqr",
    "mileage_outlier_iqr",
    "power_outlier_iqr",
    "logical_issue",
]
LEAKAGE_COLUMNS = ["price_label", "price_label_id", "top_price"]


def select_processed_csv(required_columns, preferred_filename):
    preferred_paths = [
        ("Preferred full classroom sample CSV (local)", processed_folder / preferred_filename),
        ("Repo-visible fallback sample", sample_processed_folder / preferred_filename),
    ]
    rejected = []

    for source_label, candidate in preferred_paths:
        if candidate.exists():
            missing, error = csv_missing_required_columns(candidate, required_columns)
            if not missing and error is None:
                return candidate, source_label
            rejected.append((source_label, candidate, missing, error))

    candidate = newest_csv_in_folder(processed_folder)
    if candidate is not None:
        missing, error = csv_missing_required_columns(candidate, required_columns)
        if not missing and error is None:
            return candidate, "Latest local processed CSV"
        rejected.append(("Latest local processed CSV", candidate, missing, error))

    candidate = newest_data_csv_with_required_columns(PROJECT_ROOT / "data", required_columns)
    if candidate is not None:
        return candidate, "Last-resort scan for any CSV with required columns"

    details = []
    for source_label, path, missing, error in rejected:
        if error:
            details.append(f"{source_label}: {relative_path(path)} could not be read ({error})")
        else:
            details.append(f"{source_label}: {relative_path(path)} missing {missing}")
    raise FileNotFoundError(
        "No processed CSV with the required classification columns was found. "
        "Run Notebook 02 to create data/processed/*.csv, or use data/sample/processed/.\n"
        + "\n".join(details)
    )


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(numeric_columns, categorical_columns, scale_numeric=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_transformer = Pipeline(numeric_steps)
    transformers = [("numeric", numeric_transformer, numeric_columns)]

    if categorical_columns:
        categorical_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_one_hot_encoder()),
        ])
        transformers.append(("categorical", categorical_transformer, categorical_columns))

    return ColumnTransformer(transformers=transformers)


def safe_roc_auc(y_true, y_score):
    if y_score is None or pd.Series(y_true).nunique() < 2:
        return np.nan
    try:
        return roc_auc_score(y_true, y_score)
    except ValueError:
        return np.nan


def evaluate_classifier(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    return {
        "model_name": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": safe_roc_auc(y_test, y_score),
        "predictions": y_pred,
        "probabilities": y_score,
    }


def plot_confusion_matrix(cm, title):
    fig, ax = plt.subplots(figsize=(4.5, 4))
    image = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks([0, 1], labels=["not top", "top"])
    ax.set_yticks([0, 1], labels=["not top", "top"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            ax.text(col, row, int(cm[row, col]), ha="center", va="center", color="black")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def build_threshold_table(y_true, y_score, thresholds):
    rows = []
    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "selected_count": int(y_pred.sum()),
            "selected_rate": float(y_pred.mean()),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
        })
    return pd.DataFrame(rows)


def get_feature_names(preprocessor):
    try:
        names = preprocessor.get_feature_names_out()
        return [name.replace("numeric__", "").replace("categorical__", "") for name in names]
    except Exception:
        return numeric_features + categorical_features


PROJECT_ROOT = find_repo_root(Path.cwd())
PROJECT_CONFIG = load_project_config(PROJECT_ROOT / "config" / "project_config.yaml")
RANDOM_SEED = int(PROJECT_CONFIG.get("random_state", 42))
np.random.seed(RANDOM_SEED)

make_scope = str(PROJECT_CONFIG.get("make", "Audi")).strip()
model_scope = str(PROJECT_CONFIG.get("model", "A3")).strip()
country_scope = str(PROJECT_CONFIG.get("country", "Germany")).strip()
TAG = f"{make_scope}_{model_scope}_{country_scope}".lower().replace(" ", "_")

processed_folder = PROJECT_ROOT / str(PROJECT_CONFIG.get("processed_data_path", "data/processed"))
sample_processed_folder = PROJECT_ROOT / "data" / "sample" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

selected_csv_path, selected_source = select_processed_csv(REQUIRED_COLUMNS, SESSION_07_FULL_CSV)
df_raw = pd.read_csv(selected_csv_path)
row_count_before = len(df_raw)
missing_required = [column for column in REQUIRED_COLUMNS if column not in df_raw.columns]
if missing_required:
    raise ValueError("Missing required classification columns: " + ", ".join(missing_required))

missing_optional = [column for column in OPTIONAL_COLUMNS if column not in df_raw.columns]
df = df_raw.copy()
created_listing_id = "listing_id" not in df.columns
had_top_price = "top_price" in df.columns

for column in ["mileage_km", "age_years", "power_hp", "price_eur", "actual_price_eur"]:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

df["price_label"] = df["price_label"].astype("string").str.strip().str.lower()
top_price_from_label = df["price_label"].eq("top-price").astype(int)

if had_top_price:
    top_price_text = df["top_price"].astype("string").str.strip().str.lower()
    top_price_text_map = top_price_text.map({"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0})
    existing_top_price = pd.to_numeric(df["top_price"], errors="coerce")
    existing_top_price = existing_top_price.where(existing_top_price.notna(), top_price_text_map)
    invalid_existing_mask = existing_top_price.notna() & ~existing_top_price.isin([0, 1])
    if invalid_existing_mask.any():
        raise ValueError(
            "Existing top_price contains values other than 0/1 for "
            f"{int(invalid_existing_mask.sum())} rows. Fix the target definition before modeling."
        )
    mismatch_mask = existing_top_price.notna() & existing_top_price.astype("Int64").ne(top_price_from_label)
    if mismatch_mask.any():
        raise ValueError(
            "Existing top_price does not match price_label == 'top-price' for "
            f"{int(mismatch_mask.sum())} rows. Fix the target definition before modeling."
        )
    top_price_status = "present and validated against price_label"
else:
    top_price_status = "created from price_label"

df["top_price"] = top_price_from_label

valid_modeling_rows = (
    df["price_label"].notna()
    & df["price_eur"].gt(0)
    & df["mileage_km"].gt(0)
    & df["age_years"].ge(0)
    & df["power_hp"].gt(0)
)
rows_removed = int((~valid_modeling_rows).sum())
df = df.loc[valid_modeling_rows].reset_index(drop=True)
if created_listing_id:
    df.insert(0, "listing_id", np.arange(1, len(df) + 1))
    missing_optional = [column for column in missing_optional if column != "listing_id"]

if df.empty:
    raise ValueError("The selected CSV has no valid modeling rows after filtering.")
if df["top_price"].nunique() < 2:
    raise ValueError("The target top_price has only one class after filtering.")

numeric_features_without_price = ["mileage_km", "age_years", "power_hp"]
numeric_features = ["price_eur"] + numeric_features_without_price
categorical_features = []
if "fuel_type" in df.columns and df["fuel_type"].nunique(dropna=True) > 1:
    categorical_features.append("fuel_type")
feature_columns_without_price = numeric_features_without_price + categorical_features
feature_columns = numeric_features + categorical_features
excluded_feature_columns = [column for column in LEAKAGE_COLUMNS + ["actual_price_eur"] if column in df.columns]

print("Selected CSV:", relative_path(selected_csv_path))
print("Source priority:", selected_source)
print("Configured scope:", make_scope, model_scope, country_scope)
print("Rows before filtering:", row_count_before)
print("Rows after filtering:", len(df))
print("Rows removed by required-value filter:", rows_removed)
print("Target definition: top_price = 1 when price_label == 'top-price', otherwise 0")
print("top_price status:", top_price_status)
print("Main feature columns with observable price:", feature_columns)
print("Comparison feature columns without price_eur:", feature_columns_without_price)
print("Excluded from model features because of leakage or duplicate BI naming:", excluded_feature_columns)
if missing_optional:
    print("Optional columns not found in this CSV:", ", ".join(missing_optional))

print("\nFirst 5 rows:")
display(df.head())

print("Missing values summary:")
missing_summary = df.isna().sum().sort_values(ascending=False).to_frame("missing_values")
display(missing_summary[missing_summary["missing_values"] > 0])


## Desafío 1: distribución de etiquetas y línea de base ingenua

**Tiempo sugerido:** 25 minutos

**Tarea del estudiante con funciones de ayuda**

**Razón comercial:**  
La clasificación comienza con la etiqueta. Un modelo no se puede interpretar si no se comprende la clase positiva. Un clasificador también debe superar una línea de base simple.

**Objetivo:**  
Inspeccionar acciones de clase positiva y evaluar una base de referencia de clase mayoritaria.

**Formato de salida esperado:**
- Tabla de frecuencias `price_label`.
- `top_price` tabla de recuento y porcentaje.
- Gráfico de barras de distribución de clases.
- Métricas de referencia de clase mayoritaria y matriz de confusión.

**Pausa de discusión:**
- ¿Qué tan común es la clase positiva?
- ¿Por qué la precisión puede ser engañosa?
- ¿Qué sucede si la cola de revisión omite todos los aspectos positivos?


In [ ]:
# Desafío 1 andamio guiado.
# Se proporcionan las funciones de ayuda; su tarea es inspeccionar e interpretar los resultados.

# Tu código aquí
# precio_label_summary = ...
# resumen_destino = ...
# mostrar(precio_etiqueta_summary)
# mostrar(objetivo_summary)
#
# X = df[columnas_características].copiar()
# X_sin_precio = df[feature_columns_ without_price].copiar()
# y = df["top_price"].astype(int)
# índice_tren, índice_prueba = tren_prueba_split(
#     df.índice,
#     tamaño_prueba = 0,20,
#     estado_aleatorio = SEMILLA_ALEATORIA,
#     estratificar = y,
# )
# X_tren = X.loc[tren_índice]
# X_prueba = X.loc[test_index]
# X_tren_sin_precio = X_sin_precio.loc[índice_tren]
# X_prueba_sin_precio = X_sin_precio.loc[índice_prueba]
# y_tren = y.loc[índice_tren]
# y_test = y.loc[índice_prueba]
#
# clase_mayoría = y_train.mode()[0]
# y_pred_baseline = np.repeat(clase_mayoritaria, len(y_test))
# línea_base_cm = matriz_confusión(y_test, y_pred_baseline, etiquetas=[0, 1])
# plot_confusion_matrix(baseline_cm, "Línea base de clase mayoritaria")


### Su etiqueta e interpretación de referencia

1. Participación de clase positiva:
2. Precisión de referencia de clase mayoritaria:
3. ¿La línea de base detectó positivos?
4. ¿Qué métrica hace visible la falla inicial?


## Desafío 2: Regresión logística como primer clasificador

**Tiempo sugerido:** 40 minutos

**Tarea del estudiante**

**Razón comercial:**  
A professional data scientist starts with a transparent model before moving to flexible models.

**Objetivo:**  
Entrene la regresión logística utilizando el conjunto principal de características observables, incluido `price_eur`. Para discusión, compárelo con el mismo modelo excluyendo `price_eur`.

**Conjunto de características principales:**
- `price_eur`
- `mileage_km`
- `age_years`
- `power_hp`
- `fuel_type` si está disponible

**Conjunto de funciones de comparación:**
- los mismos campos, pero sin `price_eur`

**No incluye:** `price_label`, `price_label_id`, `top_price` o `actual_price_eur`.

**Formato de salida esperado:**
- Tabla de métricas comparando con y sin `price_eur`.
- Matriz de confusión para el modelo principal con `price_eur`.
- ROC-AUC si está disponible.
- Breve interpretación de precisión y recuperación.

**Demostración dirigida por un instructor:** Los coeficientes se pueden mostrar si el tiempo lo permite, pero no son la tarea principal de codificación.


In [ ]:
# Desafío 2 andamio.
# Utilice los ayudantes compartidos de preprocesamiento y evaluación.
# use_class_weight es un indicador booleano: verdadero cuando la clase minoritaria está por debajo del 30%.
# Le dice al modelo que preste más atención a la clase más pequeña.

# Tu código aquí
# minority_share = y_train.value_counts(normalize=True).min()
# use_class_weight = minoría_share < 0,30
# logreg_class_weight = "equilibrado" si use_class_weight en caso contrario Ninguno
# logreg_pipeline_ without_price = Tubería([
#     ("preproceso", build_preprocessor(numeric_features_ without_price, categorical_features, scale_numeric=True)),
#     ("modelo", LogisticRegression(max_iter=2000, class_weight=logreg_class_weight)),
# ])
# logreg_pipeline = Tubería([
#     ("preproceso", build_preprocessor(numeric_features, categorical_features, scale_numeric=True)),
#     ("modelo", LogisticRegression(max_iter=2000, class_weight=logreg_class_weight)),
# ])
# logreg_result_sin_precio = evaluar_clasificador(
#     "Regresión logística sin precio_eur",
#     logreg_pipeline_sin_precio,
#     X_tren_sin_precio,
#     X_prueba_sin_precio,
#     y_tren,
#     y_prueba,
# )
# logreg_result = evalua_classifier("Regresión logística con precio_eur", logreg_pipeline, X_train, X_test, y_train, y_test)
# logreg_metrics_df = pd.DataFrame([
#     {k: v para k, v en logreg_result_ without_price.items() si k no está en ["predicciones", "probabilidades"]},
#     {k: v para k, v en logreg_result.items() si k no está en ["predicciones", "probabilidades"]},
# ])
# mostrar(logreg_metrics_df)
# plot_confusion_matrix(confusion_matrix(y_test, logreg_result["predicciones"], etiquetas=[0, 1]), "Regresión logística con precio_eur")
#
# Estas variables alternativas hacen que la ruta en vivo mínima viable sea ejecutable si se omite el Desafío 3.
# resultados_modelo = {
#     "Regresión logística sin precio_eur": logreg_result_sin_precio,
#     "Regresión logística con precio_eur": logreg_result,
# }
# artefactos_modelo = {
#     "Regresión logística sin precio_eur": {"pipeline": logreg_pipeline_sin_precio, "X_all": X_sin_precio},
#     "Regresión logística con precio_eur": {"pipeline": logreg_pipeline, "X_all": X},
# }
# selected_model_name = "Regresión logística con precio_eur" # respaldo si se omite el desafío 3


### Su interpretación de la regresión logística

1. Precisión:
2. Recuerde:
3. ¿Es útil el modelo como filtro de primera revisión?
4. ¿Qué costo comercial aparece si el retiro es demasiado bajo?


## Desafío 3: comparación de un modelo flexible

**Tiempo sugerido:** 35 minutos

**Tarea del estudiante**

**Razón comercial:**  
Los modelos flexibles pueden capturar patrones no lineales, pero pueden ser más difíciles de explicar.

**Objetivo:**  
Compare la regresión logística con un modelo flexible: Random Forest. Mantenga visibles las variantes con precio y sin precio para que se pueda discutir la elección de la función.

**Formato de salida esperado:**
- Tabla comparativa.
- Matrices de confusión.
- Cuadro de importancia de características de Random Forest opcional.

**Pausa de discusión:**
- ¿Cuánto cambia `price_eur` en precisión, recuperación, F1 y ROC-AUC?
- ¿Qué modelo es mejor para una cola de revisión?
- ¿Qué métrica respalda esa decisión?
- ¿La ganancia vale la complejidad?


In [ ]:
# Desafío 3 andamio.
# Compare la regresión logística con el bosque aleatorio solo en la ruta principal.
# use_class_weight debería provenir del Desafío 2.
# model_results debe ser un diccionario que almacene cada resultado de modelo por nombre de modelo.

# Tu código aquí
# forest_class_weight = "submuestra_equilibrada" si use_class_weight más Ninguno
# rf_pipeline_ without_price = Tubería([
#     ("preproceso", build_preprocessor(numeric_features_ without_price, categorical_features, scale_numeric=False)),
#     ("modelo", RandomForestClassifier(
#         n_estimadores=100,
#         profundidad_max=8,
#         estado_aleatorio = SEMILLA_ALEATORIA,
#         n_trabajos=-1,
#         class_weight=bosque_class_weight,
#     )),
# ])
# rf_pipeline = Tubería([
#     ("preproceso", build_preprocessor(numeric_features, categorical_features, scale_numeric=False)),
#     ("modelo", RandomForestClassifier(
#         n_estimadores=100,
#         profundidad_max=8,
#         estado_aleatorio = SEMILLA_ALEATORIA,
#         n_trabajos=-1,
#         class_weight=bosque_class_weight,
#     )),
# ])
# rf_result_sin_precio = evaluar_clasificador(
#     "Bosque aleatorio sin precio_eur",
#     rf_pipeline_sin_precio,
#     X_tren_sin_precio,
#     X_prueba_sin_precio,
#     y_tren,
#     y_prueba,
# )
# rf_result = evalua_classifier("Bosque aleatorio con precio_eur", rf_pipeline, X_train, X_test, y_train, y_test)
# resultados_modelo = {
#     "Regresión logística sin precio_eur": logreg_result_sin_precio,
#     "Regresión logística con precio_eur": logreg_result,
#     "Bosque aleatorio sin precio_eur": rf_result_sin_precio,
#     "Bosque aleatorio con precio_eur": rf_result,
# }
# artefactos_modelo = {
#     "Regresión logística sin precio_eur": {"pipeline": logreg_pipeline_sin_precio, "X_all": X_sin_precio},
#     "Regresión logística con precio_eur": {"pipeline": logreg_pipeline, "X_all": X},
#     "Bosque aleatorio sin precio_eur": {"pipeline": rf_pipeline_sin_precio, "X_all": X_sin_precio},
#     "Bosque aleatorio con precio_eur": {"pipeline": rf_pipeline, "X_all": X},
# }
# Compare todos los resultados utilizando precisión, recuperación, F1 y ROC-AUC.


### Tu decisión de comparación de modelos

1. Modelo seleccionado para la cola de revisión:
2. Evidencia métrica:
3. Compensación entre desempeño y explicabilidad:


## Desafío 4: Política de umbral y capacidad de revisión

**Tiempo sugerido:** 35 minutos

**Tarea del estudiante y pausa de discusión**

**Razón comercial:**  
Un clasificador genera probabilidades. El umbral es una decisión operativa, no una verdad modelo.

**Objetivo:**  
Analice cómo los diferentes umbrales de probabilidad cambian la precisión, la recuperación, la F1 y la cantidad de listados seleccionados. Si se omitió el Desafío 3, utilice la regresión logística de `model_results` para el análisis de umbral.

**Formato de salida esperado:**
- Tabla de umbrales para 0,30, 0,40, 0,50, 0,60, 0,70.
- Gráfico de umbral versus precisión y recuperación.
- Umbral recomendado o política Top-N.

**Pausa de discusión:**
- Los falsos positivos consumen tiempo del analista.
- Los falsos negativos pierden oportunidades.
- 0,50 no es automáticamente correcto.
- Umbral es una política operativa para la capacidad de revisión.


In [ ]:
# Desafío 4 andamio.
# Seleccione el modelo que usaría para una cola de revisión y luego pruebe los umbrales de políticas.
# selected_model_name debe ser una clave de model_results, por ejemplo, "Bosque aleatorio con precio_eur".
# Si se omitió el Desafío 3, mantenga selected_model_name = "Regresión logística con precio_eur" del Desafío 2.
# Se elegirá umbral_recomendado después de inspeccionar la tabla de umbrales.

# Tu código aquí
# selected_model_name = "Bosque aleatorio con precio_eur" # U otro modelo, según su decisión del Desafío 3.
# resultado_seleccionado = resultados_modelo[nombre_modelo_seleccionado]
# umbral_análisis_df = build_threshold_table(
#     y_prueba,
#     resultado_seleccionado["probabilidades"],
#     umbrales = [0,30, 0,40, 0,50, 0,60, 0,70],
# )
# visualización (umbral_análisis_df)
# umbral_recomendado = 0,50


### Su recomendación de política de umbral

1. Umbral recomendado o regla Top-N:
2. Volumen de revisión esperado:
3. Principal compensación comercial:
4. Por qué 0,50 no es automáticamente correcto:


## Desafío 5: cola de revisión de clasificación lista para BI

**Tiempo sugerido:** 20 minutos

**Demostración dirigida por un instructor con interpretación del estudiante**

**Razón comercial:**  
Un modelo de clasificación resulta útil sólo cuando crea un artefacto de revisión procesable.

**Objetivo:**  
Ajuste el modelo seleccionado en todas las filas válidas, genere `probability_top_price` y cree `classification_review_queue`.

**Formato de salida esperado:**
- `probability_top_price`
- `predicted_top_price`
- `classification_review_queue`
- Salidas CSV guardadas en `reports/`

**Límite operativo importante:**  
En la evaluación histórica, está bien mantener `price_label` y `top_price` en la cola de revisión para su validación. En una puntuación futura real, `price_label` y `top_price` no se conocerían y no se debería confiar en ellos operativamente.

**Pausa de discusión:**
- ¿Qué listados merecen la revisión de los analistas primero?
- ¿Qué predicciones podrían ser artefactos del modelo?
- ¿Cómo debería mostrar Power BI esta salida?


In [ ]:
# Desafío 5 andamio.
# El instructor puede ejecutar esto después de la discusión previa.
# selected_model_name proviene del Desafío 4.
# umbral_recomendado es el umbral operativo seleccionado en la discusión sobre la política de umbrales.

# Tu código aquí
# artefacto_final = artefactos_modelo[nombre_modelo_seleccionado]
# final_model = final_artifact["tubería"]
# final_X = final_artifact["X_all"]
# modelo_final.fit(final_X, y)
# probabilidad_precio_superior = modelo_final.predict_proba(final_X)[:, 1]
# predicted_top_price = (probabilidad_precio_superior >= umbral_recomendado).astype(int)
#
# Cree Classification_review_queue con columnas de contexto.
# Ahorrar:
# - informes/{TAG}_classification_review_queue.csv
# - informes/{TAG}_classification_model_comparison.csv
# - informes/{TAG}_classification_threshold_analysis.csv


### Su conclusión de transferencia BI

1. Tres listados para revisión de analistas:
2. Riesgo de un modelo:
3. Un insumo empresarial que falta antes de tomar una decisión real:
4. ¿Qué debería mostrar Power BI?


## Extensión opcional: modelos flexibles adicionales y calibración de probabilidad

**Extensión opcional**

Úselo solo si la clase avanza más rápido de lo esperado o como práctica después de clase.

Decision Tree y HistGradientBoosting se pueden analizar como modelos flexibles adicionales, pero no forman parte de la ruta principal en vivo. Random Forest es suficiente para enseñar el paso de un modelo transparente a un modelo flexible.

La calibración pregunta si las probabilidades predichas coinciden con las frecuencias observadas. Por ejemplo, entre los listados con una puntuación cercana a 0,80, ¿aproximadamente el 80% pertenece realmente a la clase positiva?

Esto es diferente de la selección de umbral. La selección de umbral elige una regla operativa para la capacidad de revisión. La calibración evalúa si la escala de probabilidad en sí es confiable.

No agregue calibración a la ruta del núcleo. Si se usa, discútalo conceptualmente o demuéstralo después de clase.


## Discusión final: ¿qué aprendimos sobre la clasificación como capa de priorización?

Utilice estas indicaciones para la discusión final en clase:

- ¿Por qué inspeccionamos el objetivo antes del entrenamiento?
- ¿Por qué construimos una base ingenua?
- ¿Por qué es peligrosa la precisión en una clasificación desequilibrada?
- ¿Por qué se permite `price_label` como origen del destino pero no como característica?
- ¿Por qué se permite `price_eur` como característica?
- ¿Qué es la fuga?
- ¿Cómo se relacionan la precisión y la recuperación con los costos comerciales?
- ¿Por qué la selección del umbral es una decisión política?
- ¿Cómo se convierte `probability_top_price` en una capa de priorización lista para BI?

Mensaje final:

**Un clasificador útil no termina en una métrica. Termina en una capa de probabilidad que ayuda a las personas a priorizar la revisión, comprender las compensaciones y tomar mejores decisiones.**
